# Lahore UC

## Import Libraries and initialisation

In [7]:
import ee, geemap, geopandas as gpd, pandas as pd
import folium

ee.Authenticate()
ee.Initialize()

### Read shape files

In [3]:

gdf = gpd.read_file("../../data/Lahore UCs/Lahore UC.shp")

# Optional: restrict to Lahore district only if present
if "DISTRICT" in gdf.columns:
    gdf = gdf[gdf["DISTRICT"].astype(str).str.contains("Lahore", case=False, na=False)]

if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)

if "uc_id" not in gdf.columns:
    print("Adding 'uc_id' column as unique identifier.")
    gdf["uc_id"] = range(1, len(gdf) + 1)

ucs = geemap.geopandas_to_ee(gdf) # <-- ee.FeatureCollection in memory

Adding 'uc_id' column as unique identifier.


Use cell below if extracting from uploaded shape file on GEE directly

In [3]:
uc_asset = "projects/ee-ahmedabclr35/assets/Lahore_union_Council_Boundries"
ucs = ee.FeatureCollection(uc_asset)

## Start and end date

In [4]:
start = "2025-01-01"
end = "2025-08-30"

# 1) NDVI 

In [ ]:
from ndvi import compute_uc_ndvi

results = compute_ndvi(ucs, start, end)


Generating URL ...
Please wait ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/GEE/NDVI.geojson
GeoJSON saved to NDVI.geojson
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/GEE/NDVI.geojson
GeoJSON saved to NDVI.geojson


### Plot on folium

In [ ]:
import numpy as np, geopandas as gpd, folium
uc_gdf = gpd.read_file("NDVI.geojson")
val_col = "NDVI" if "NDVI" in uc_gdf.columns else ("mean" if "mean" in uc_gdf.columns else None)
if val_col is None:
    raise ValueError("No NDVI column found in NDVI.geojson; expected 'NDVI' or 'mean'.")
uc_gdf = uc_gdf.dropna(subset=[val_col])
if uc_gdf.crs and uc_gdf.crs.to_epsg() != 4326:
    uc_gdf = uc_gdf.to_crs(4326)
center = [uc_gdf.geometry.centroid.y.mean(), uc_gdf.geometry.centroid.x.mean()]
m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")
vmin, vmax = float(uc_gdf[val_col].min()), float(uc_gdf[val_col].max())
if np.isfinite(vmin) and np.isfinite(vmax) and vmin != vmax:
    bins = list(np.quantile(uc_gdf[val_col], [0, 0.2, 0.4, 0.6, 0.8, 1]))
else:
    bins = list(np.linspace(vmin, vmax if vmin != vmax else vmin + 1e-6, 6))
folium.Choropleth(
    geo_data=uc_gdf, data=uc_gdf,
    columns=["UC", val_col], key_on="feature.properties.UC",
    fill_color="YlGn", fill_opacity=0.7, line_opacity=0.2,
    bins=bins, nan_fill_opacity=0, legend_name="Mean NDVI",
).add_to(m)
folium.GeoJson(
    uc_gdf,
    style_function=lambda x: {"fillColor": "transparent", "color": "black", "weight": 0.3},
    popup=folium.GeoJsonPopup(fields=["UC", val_col], aliases=["UC Name", "NDVI Mean"], localize=True)
).add_to(m)
m.save("NDVI_heatmap.html")
m

/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_48532/3944028875.py:12: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]


# 2) Nightlights

In [10]:
import importlib
from nl import compute_nl
import nl
importlib.reload(nl)



nl_results = compute_nl(ucs, start, end)

Generating URL ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/GEE/NL.geojson
Wrote NL.geojson
Wrote NL.csv
Completed UC Nightlights aggregation.


### Folium plot

In [11]:
import numpy as np, geopandas as gpd, folium
gdf = gpd.read_file("NL.geojson")
val_col = "avg_rad" if "avg_rad" in gdf.columns else ("mean" if "mean" in gdf.columns else None)
if val_col is None:
    raise ValueError("No nightlights column found in NL.geojson; expected 'avg_rad' or 'mean'.")
gdf = gdf.dropna(subset=[val_col])
if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)
center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]
m_nl = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")
vmin, vmax = float(gdf[val_col].min()), float(gdf[val_col].max())
if np.isfinite(vmin) and np.isfinite(vmax) and vmin != vmax:
    bins = list(np.quantile(gdf[val_col], [0, 0.2, 0.4, 0.6, 0.8, 1]))
else:
    bins = list(np.linspace(vmin, vmax if vmin != vmax else vmin + 1e-6, 6))
folium.Choropleth(
    geo_data=gdf, data=gdf,
    columns=["UC", val_col], key_on="feature.properties.UC",
    fill_color="YlOrRd", fill_opacity=0.7, line_opacity=0.2,
    bins=bins, nan_fill_opacity=0, legend_name="VIIRS Nightlights Mean",
).add_to(m_nl)
folium.GeoJson(
    gdf,
    style_function=lambda x: {"fillColor": "transparent", "color": "black", "weight": 0.3},
    tooltip=folium.GeoJsonTooltip(fields=["UC", val_col], aliases=["UC Name:", "Nightlights Mean:"], localize=True)
).add_to(m_nl)
m_nl.save("NL_heatmap.html")

/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_49554/742241416.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]


## LPPI (Land-Pollution Proxy Index LPPI)

In [4]:
from lppi import compute_lppi
lppi_results = compute_lppi(ucs, start, end, include_osm=True, export_path="LPPI.geojson")


Generating URL ...
Please wait ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/GEE/LPPI.geojson
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/GEE/LPPI.geojson


In [5]:
import geopandas as gpd, folium, numpy as np
gdf = gpd.read_file("LPPI.geojson").dropna(subset=["mean"])
if gdf.crs and gdf.crs.to_epsg()!=4326:
    gdf = gdf.to_crs(4326)
center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]
m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")
bins = list(np.linspace(float(gdf["mean"].min()), float(gdf["mean"].max()), 7))
folium.Choropleth(
    geo_data=gdf, data=gdf,
    columns=["UC","mean"], key_on="feature.properties.UC",
    fill_color="YlOrRd", fill_opacity=0.7, line_opacity=0.2,
    bins=bins, legend_name="Land Pollution Proxy (LPPI, higher=worse)",
).add_to(m)
folium.GeoJson(
    gdf,
    style_function=lambda x: {"fillColor": "transparent", "color": "black", "weight": 0.3},
    popup=folium.GeoJsonPopup(fields=["UC","mean"], aliases=["UC","LPPI"])
).add_to(m)
m.save("LPPI_heatmap.html")
m

/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_71309/1990949563.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]


## Building Footprint

In [19]:
# Import the building footprint module
import sys
import importlib

# Force reload to ensure we get latest changes
if 'bfp' in sys.modules:
    del sys.modules['bfp']

import bfp
from bfp import compute_building_footprint, compute_building_footprint_stats



# Compute building footprint metrics for all Union Councils
# Note: For large regions like Lahore (1.2M buildings), GeoJSON export times out
# Solution: Export CSV only (~10 seconds), then merge with shapefile in next cell
uc_buildings = compute_building_footprint(
    ucs,
    building_min_conf=0.75,          # Confidence threshold (0-1)
    use_exact_intersection=False,    # Set to True for more accurate area calculations (slower)
    export_path="UC_Buildings_OpenBuildings.geojson",
    export_geojson=False             # CSV only - no timeout! ✅
)

print("\n✅ CSV exported successfully!")
print("💡 Next: Run the cell below to merge CSV with UC shapefile")


🔄 Module reloaded successfully
📍 Using bfp from: /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/GEE/bfp.py
🏢 Computing building footprint metrics...
   Confidence threshold: 0.75
   Exact intersection: False
   Processing Union Councils...
   Exporting results...
   Creating UC_Buildings_OpenBuildings.csv...
Generating URL ...
Generating URL ...
Please wait ...
Please wait ...
An error occurred while downloading.


ValueError: HTTPSConnectionPool(host='earthengine.googleapis.com', port=443): Read timed out. (read timeout=300)

### 📋 Diagnostic: Check BFP Module Status

Run this cell first to verify the module is loaded correctly.

### Visualize Building Footprint Results

### Merge Building Metrics with UC Boundaries (if GeoJSON export failed)

In [ ]:
import geopandas as gpd, folium, numpy as np

# Load exported building metrics
gdf = gpd.read_file("UC_Buildings_OpenBuildings.geojson").dropna(subset=["bld_coverage_pct"])
if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)

center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]
bins = list(np.linspace(float(gdf["bld_coverage_pct"].min()), float(gdf["bld_coverage_pct"].max()), 7))

m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")
folium.Choropleth(
    geo_data=gdf, data=gdf,
    columns=["UC", "bld_coverage_pct"], key_on="feature.properties.UC",
    fill_color="YlGnBu", fill_opacity=0.7, line_opacity=0.2,
    bins=bins, legend_name="Building Coverage (%)",
).add_to(m)

folium.GeoJson(
    gdf,
    style_function=lambda x: {"fillColor": "transparent", "color": "black", "weight": 0.3},
    popup=folium.GeoJsonPopup(
        fields=["UC", "bld_count", "bld_coverage_pct"],
        aliases=["UC", "Building Count", "Coverage (%)"]
    )
).add_to(m)

m.save("UC_Buildings_heatmap.html")
m

## High-rise Buildings

In [86]:

# ----------------- config -----------------
YEAR = 2023
HR_MIN_M = 30        # try 25 if too sparse
PRESENCE_MIN = 0.6   # try 0.40–0.45 if too many zeros
SCALE_M = 4
TILESCALE = 10       # bump to 12 if needed

# --- helper to coalesce keys (plain vs *_sum) and default to 0 ---
def _get_num(f, primary, alt):
    names = f.propertyNames()
    return ee.Number(
        ee.Algorithms.If(
            names.contains(primary), f.get(primary),
            ee.Algorithms.If(names.contains(alt), f.get(alt), 0)
        )
    )

# ----------------- data -------------------
col = (ee.ImageCollection("GOOGLE/Research/open-buildings-temporal/v1")
         .filterBounds(ucs.geometry())
         .filterDate(f"{YEAR}-01-01", f"{YEAR+1}-01-01"))

img      = col.mosaic()
height   = img.select("building_height")            # meters
presence = img.select("building_presence")          # 0..1 (uncalibrated)
frac     = img.select("building_fractional_count")  # expected count density

# masks
build_mask = presence.gte(PRESENCE_MIN)
hr_mask    = build_mask.And(height.gte(HR_MIN_M))

# per-pixel metrics  ✅ NO pixelArea on counts
bld_count_est = frac.updateMask(build_mask).rename("bld_count_est")
hr_count_est  = frac.updateMask(hr_mask).rename("hr_count_est")
hr_area_m2    = ee.Image.pixelArea().updateMask(hr_mask).rename("hr_area_m2")

# stack and reduce (sum over each UC) — simple & robust
metrics = ee.Image.cat([bld_count_est, hr_count_est, hr_area_m2])
uc_sums = metrics.reduceRegions(
    collection=ucs,
    reducer=ee.Reducer.sum(),   # EE may output plain band names or *_sum
    scale=SCALE_M,
    tileScale=TILESCALE
)

# add derived fields and metadata; KEEP GEOMETRY
def add_derived(f):
    f = ee.Feature(f)
    total = _get_num(f, "bld_count_est", "bld_count_est_sum")
    hr    = _get_num(f, "hr_count_est",  "hr_count_est_sum")
    hra   = _get_num(f, "hr_area_m2",    "hr_area_m2_sum")

    area_km2 = f.geometry().area(1).divide(1e6)
    dens  = ee.Algorithms.If(area_km2.gt(0), hr.divide(area_km2), 0)
    share = ee.Algorithms.If(total.gt(0),    hr.divide(total).multiply(100), 0)

    return f.set({
        "year": YEAR,
        "highrise_min_m": HR_MIN_M,
        "presence_min": PRESENCE_MIN,
        "bld_count_est": total,     # overwrite with coalesced numeric
        "hr_count_est":  hr,
        "hr_area_m2":    hra,
        "hr_density_km2": dens,
        "hr_share_pct":   share
    })

uc_out = uc_sums.map(add_derived)

# choose the props to keep (adjust to your UC schema)
keep = ["UC","uc_id","year","highrise_min_m","presence_min",
        "bld_count_est","hr_count_est","hr_share_pct","hr_area_m2","hr_density_km2"]
uc_out_slim = uc_out.select(keep)

# Export to CSV (CSV ignores geometry; GeoJSON will include it)
csv_path = f"HR_{YEAR}_min{HR_MIN_M}m.csv"
geemap.ee_export_vector(uc_out_slim, filename=csv_path)
geojson_path = csv_path.replace(".csv", ".geojson")
geemap.ee_export_vector(uc_out_slim, filename=geojson_path)
# Optional quick sanity print (client-side)
df = pd.read_csv(csv_path)
cols = [c for c in ["UC","hr_count_est","hr_density_km2","hr_share_pct"] if c in df.columns]
if cols:
    print(df[cols].sort_values("hr_count_est", ascending=False).head(10))
print(f"Saved {csv_path}")


Generating URL ...
Please wait ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/GEE/HR_2023_min30m.csv
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/GEE/HR_2023_min30m.csv
Generating URL ...
Generating URL ...
Please wait ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/GEE/HR_2023_min30m.geojson
               UC  hr_count_est  hr_density_km2  hr_share_pct
20     Cantonment      1.263350        0.012791      0.042895
58        Gulberg      1.251192        0.346610      1.095308
16        Kamahan      1.232940        0.030378      0.234070
59       Al-hamra      0.876679        0.128528      0.584996
57  Makkah Colony      0.666134        0.341976      0.816811
21  Ali Raza Abad      0.536179        0.015497      0.049907
60    Race Course      0.477118        0.089595      0.416138
15          Barki      0.423689        0.00693

In [89]:
# Visualize High-rise Buildings on Folium Map
import geopandas as gpd, folium, numpy as np

# Load the exported GeoJSON with high-rise metrics
gdf = gpd.read_file('HR_2023_min30m.geojson')

# Choose metric to visualize (hr_density_km2 shows concentration best)
metric_col = "hr_density_km2"  # or use "hr_count_est" or "hr_share_pct"

# Drop NAs and ensure proper CRS
gdf = gdf.dropna(subset=[metric_col])
if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)

# Calculate map center
center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]

# Create bins for legend
vmin, vmax = float(gdf[metric_col].min()), float(gdf[metric_col].max())
if np.isfinite(vmin) and np.isfinite(vmax) and vmin != vmax:
    bins = list(np.quantile(gdf[metric_col], [0, 0.25, 0.5,  0.75, 1]))
else:
    bins = list(np.linspace(vmin, vmax if vmin != vmax else vmin + 1e-6, 6))

# Create Folium map
m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")

# Add choropleth layer
folium.Choropleth(
    geo_data=gdf,
    data=gdf,
    columns=["UC", metric_col],
    key_on="feature.properties.UC",
    fill_color="YlOrRd",  # Yellow to Red colormap (good for density)
    fill_opacity=0.7,
    line_opacity=0.2,
    bins=bins,
    nan_fill_opacity=0,
    legend_name=f"High-rise Density (buildings/km²)",
).add_to(m)

# Add interactive tooltips
folium.GeoJson(
    gdf,
    style_function=lambda x: {"fillColor": "transparent", "color": "black", "weight": 0.3},
    tooltip=folium.GeoJsonTooltip(
        fields=["UC", "hr_count_est", "hr_density_km2", "hr_share_pct"],
        aliases=["UC Name", "High-rise Count", "Density (per km²)", "% of Buildings"],
        localize=True
    )
).add_to(m)

# Save map
map_path = f"HR_{YEAR}_min{HR_MIN_M}m_map.html"
m.save(map_path)
print(f"✓ Saved interactive map to {map_path}")

m  # Display in notebook

✓ Saved interactive map to HR_2023_min30m_map.html


/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_7672/2032771806.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]
